## Lab 4 - N-gram Language Models (Unsmoothed, Add-1, Add-K, Interpolated, Kneser-Ney)

**Modifications made to the original notebook:**

1. **Interpolation weights are now learned from the dev set, not hard-coded.**
   - For the unigram model (N=1), the assignment's formula
     `P_smooth(w) = lambda * P(w) + (1 - lambda) * (1/V)` is used, with `lambda`
     chosen by a grid search that maximizes dev-set log-likelihood.
   - For N=2, 3, 4, standard Jelinek-Mercer interpolation is used:
     `P(w | context) = sum_{k=1}^{N} lambda_k * P_k(w | last k-1 words)`.
     All `lambda_k` are found via a grid search over the dev set and are
     **explicitly normalized so they sum to 1**.
   - The learned weights are printed during the run and saved to
     `interpolation_weights.csv`.
2. **Vocabulary size (V) fix:** `<s>` is no longer counted in the vocabulary
   used for smoothing, since it is only ever context/history padding and is
   never a token a model predicts. Including it previously inflated every
   Laplace / Add-K / interpolation denominator.
3. **Kneser-Ney floor fix:** a word that never appears as a continuation of
   another word in training previously got probability 0 under Kneser-Ney,
   which could zero out an entire probability chain and report infinite
   perplexity. It now falls back to a uniform floor of `1/V`.
4. **ZIP fix:** only the generated `*.csv` result files are archived. The
   original `train.txt` / `dev.txt` / `test.txt` corpus files were
   previously being swept into the ZIP as well, bloating it unnecessarily.
5. Removed a couple of no-op `del` statements left over from earlier drafts.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

BASE_FOLDER = "/content/drive/MyDrive/ILLM_LAB_4"

TRAIN_FILE = os.path.join(BASE_FOLDER, "train.txt")
DEV_FILE = os.path.join(BASE_FOLDER, "dev.txt")
TEST_FILE = os.path.join(BASE_FOLDER, "test.txt")

print("Train file exists:", os.path.exists(TRAIN_FILE))
print("Dev file exists:", os.path.exists(DEV_FILE))
print("Test file exists:", os.path.exists(TEST_FILE))

print("\nTrain file:", TRAIN_FILE)
print("Dev file:", DEV_FILE)
print("Test file:", TEST_FILE)


Train file exists: True
Dev file exists: True
Test file exists: True

Train file: /content/drive/MyDrive/ILLM_LAB_4/train.txt
Dev file: /content/drive/MyDrive/ILLM_LAB_4/dev.txt
Test file: /content/drive/MyDrive/ILLM_LAB_4/test.txt


In [ ]:
import math
import re
import os
import gc
import csv
import random
import zipfile

from collections import Counter, defaultdict

import pandas as pd


# USER SETTINGS


BASE_FOLDER = "/content/drive/MyDrive/ILLM_LAB_4"

TRAIN_FILE = os.path.join(BASE_FOLDER, "train.txt")
DEV_FILE = os.path.join(BASE_FOLDER, "dev.txt")
TEST_FILE = os.path.join(BASE_FOLDER, "test.txt")

MAX_N = 4

K_VALUE = 0.5          # Add-K smoothing constant, 0 < K < 1
D_VALUE = 0.75         # Kneser-Ney discount
EPSILON = 1e-10        # Small value to prevent log(0) errors

START_TOKEN = "<s>"
END_TOKEN = "</s>"
UNK_TOKEN = "<UNK>"

# Number of rows written at one time.
# Lower this if RAM is still a problem.
CSV_CHUNK_SIZE = 50000

# How many dev sentences to use when GRID-SEARCHING interpolation weights.
# Tuning on the full million-sentence dev split is unnecessarily slow; a
# random sample gives a stable estimate of the optimal weights.
LAMBDA_TUNING_SAMPLE_SIZE = 3000
LAMBDA_TUNING_SEED = 42
LAMBDA_GRID_STEP = 0.1          # grid resolution for the N >= 2 interpolation weights
UNIGRAM_LAMBDA_GRID_STEP = 0.05  # finer grid for the single unigram lambda

ZIP_FILE = os.path.join(BASE_FOLDER, "ngram_language_model_results.zip")


# CREATE OUTPUT FOLDER


os.makedirs(BASE_FOLDER, exist_ok=True)


# TOKENIZATION


TOKEN_PATTERN = re.compile(r"\w+|[^\w\s]", re.UNICODE)


def tokenize(sentence):

    sentence = sentence.strip()
    sentence = re.sub(r"\s+", " ", sentence)

    return TOKEN_PATTERN.findall(sentence)


# READ SENTENCES


def read_sentences(filename):

    sentences = []

    with open(filename, "r", encoding="utf-8") as file:

        for line in file:

            line = line.strip()

            if not line:
                continue

            tokens = tokenize(line)

            if tokens:
                sentences.append(tokens)

    return sentences


# BUILD VOCABULARY


def build_vocabulary(sentences):
    """
    Vocabulary used for smoothing denominators (V).

    NOTE: <s> is intentionally NOT added here. <s> is only ever used to pad
    history/context - it is never a token that a model assigns probability
    to as an outcome, so it must not be counted in V.
    """

    vocabulary = set()

    for sentence in sentences:
        vocabulary.update(sentence)

    vocabulary.add(UNK_TOKEN)
    vocabulary.add(END_TOKEN)

    return vocabulary


# REPLACE UNKNOWN WORDS


def replace_unknown_words(sentences, vocabulary):

    new_sentences = []

    for sentence in sentences:

        new_sentence = []

        for word in sentence:

            if word in vocabulary:
                new_sentence.append(word)

            else:
                new_sentence.append(UNK_TOKEN)

        new_sentences.append(new_sentence)

    return new_sentences


# PREPARE ONE SENTENCE

def prepare_sentence(sentence, n):
    return ([START_TOKEN] * (n - 1) + sentence + [END_TOKEN])


# BUILD N-GRAM COUNTS
def build_ngram_counts(sentences, n):

    counts = Counter()

    for sentence in sentences:

        padded = prepare_sentence(sentence, n)

        for i in range(len(padded) - n + 1):
            ngram = tuple(padded[i:i + n])
            counts[ngram] += 1

    return counts


# BUILD HISTORY COUNTS
def build_history_counts(ngram_counts, n):

    if n == 1:
        return {(): sum(ngram_counts.values())}

    history_counts = Counter()

    for ngram, count in ngram_counts.items():
        history_counts[ngram[:-1]] += count

    return history_counts


# BUILD ONE MODEL
def build_model(train_sentences, n):

    print(f"\nBuilding N={n} model...")

    counts = build_ngram_counts(train_sentences, n)
    histories = build_history_counts(counts, n)

    print(f"Unique N={n} grams:", f"{len(counts):,}")
    print("Histories:", f"{len(histories):,}")

    return counts, histories


# UNSMOOTHED PROBABILITY
def unsmoothed_probability(ngram, counts, histories, n):

    count = counts.get(ngram, 0)

    if count == 0:
        return 0.0

    if n == 1:
        total = histories[()]

        if total == 0:
            return 0.0

        return count / total

    history = ngram[:-1]

    history_count = histories.get(history, 0)

    if history_count == 0:
        return 0.0

    return count / history_count


# ADD-ONE PROBABILITY
def add_one_probability(ngram, counts, histories, vocabulary_size, n):

    count = counts.get(ngram, 0)

    if n == 1:
        total = histories[()]
        return (count + 1) / (total + vocabulary_size)

    history = ngram[:-1]
    history_count = histories.get(history, 0)

    return (count + 1) / (history_count + vocabulary_size)


# ADD-K PROBABILITY
def add_k_probability(ngram, counts, histories, vocabulary_size, n):

    count = counts.get(ngram, 0)

    if n == 1:
        total = histories[()]

        return (count + K_VALUE) / (total + K_VALUE * vocabulary_size)

    history = ngram[:-1]
    history_count = histories.get(history, 0)

    return (count + K_VALUE) / (history_count + K_VALUE * vocabulary_size)


# -----------------------------------------------------------------------
# INTERPOLATION: LEARN THE LAMBDAS ON THE DEV SET, THEN NORMALIZE THEM
# -----------------------------------------------------------------------

def collect_component_probabilities(sentences, n, all_counts, all_histories,
                                     sample_size=None, seed=LAMBDA_TUNING_SEED):
    """
    For every n-gram token found in `sentences`, compute the raw MLE
    probability at every order 1..n. These "component" probabilities are
    reused for every candidate weight vector during the grid search, so we
    only ever touch the count dictionaries once.
    """

    if sample_size is not None and len(sentences) > sample_size:
        rng = random.Random(seed)
        sentences = rng.sample(sentences, sample_size)

    component_rows = []

    for sentence in sentences:

        padded = prepare_sentence(sentence, n)

        for i in range(len(padded) - n + 1):

            ngram = tuple(padded[i:i + n])

            components = [
                unsmoothed_probability(ngram[-k:], all_counts[k], all_histories[k], k)
                for k in range(1, n + 1)
            ]

            component_rows.append(components)

    return component_rows


def generate_simplex_grid(num_parts, step=LAMBDA_GRID_STEP):
    """
    Yield every weight vector of length `num_parts` whose entries are
    multiples of `step` and that sums to exactly 1.0 - i.e. a coarse grid
    over the probability simplex.
    """

    steps = int(round(1.0 / step))

    def recurse(remaining_parts, remaining_steps):

        if remaining_parts == 1:
            yield (remaining_steps * step,)
            return

        for value in range(remaining_steps + 1):
            for rest in recurse(remaining_parts - 1, remaining_steps - value):
                yield (value * step,) + rest

    for combo in recurse(num_parts, steps):
        yield combo


def optimize_interpolation_weights(component_rows, num_orders, step=LAMBDA_GRID_STEP):
    """
    Grid-search lambda_1 .. lambda_num_orders (order 1 = unigram, ...
    order num_orders = the highest order) on precomputed dev-set component
    probabilities. The vector that maximizes dev log-likelihood wins.

    The unigram weight (lambda_1) is required to be > 0. Since every word
    that survives unknown-word replacement is in the training vocabulary,
    its unigram MLE is always > 0, so this guarantees the interpolated
    probability of any dev/test token can never be exactly zero.
    """

    best_weights = None
    best_log_likelihood = float("-inf")

    for weights in generate_simplex_grid(num_orders, step=step):

        if weights[0] <= 0:
            continue

        log_likelihood = 0.0
        valid = True

        for components in component_rows:

            probability = sum(w * c for w, c in zip(weights, components))

            if probability <= 0:
                valid = False
                break

            log_likelihood += math.log(probability)

        if not valid:
            continue

        if log_likelihood > best_log_likelihood:
            best_log_likelihood = log_likelihood
            best_weights = weights

    if best_weights is None:
        # Extremely sparse dev data - fall back to an even split.
        best_weights = tuple([1.0 / num_orders] * num_orders)

    # Explicit normalization so the returned weights always sum to exactly 1,
    # even after any floating point drift.
    total = sum(best_weights)
    normalized_weights = tuple(w / total for w in best_weights)

    return normalized_weights, best_log_likelihood


def optimize_unigram_lambda(dev_sentences, all_counts, all_histories, vocabulary_size,
                             step=UNIGRAM_LAMBDA_GRID_STEP,
                             sample_size=LAMBDA_TUNING_SAMPLE_SIZE,
                             seed=LAMBDA_TUNING_SEED):
    """
    Grid-search the single lambda used in the assignment's unigram smoothing
    formula:  P_smooth(w) = lambda * P(w) + (1 - lambda) * (1 / V)
    lambda is chosen to maximize dev-set log-likelihood. (lambda, 1 - lambda)
    already sum to 1 by construction, i.e. they are normalized.
    """

    counts_1 = all_counts[1]
    histories_1 = all_histories[1]

    sentences = dev_sentences
    if sample_size is not None and len(sentences) > sample_size:
        rng = random.Random(seed)
        sentences = rng.sample(sentences, sample_size)

    unigram_mles = []

    for sentence in sentences:
        padded = prepare_sentence(sentence, 1)

        for token in padded:
            ngram = (token,)
            unigram_mles.append(unsmoothed_probability(ngram, counts_1, histories_1, 1))

    uniform = 1.0 / vocabulary_size

    steps = int(round(1.0 / step))
    best_lambda = None
    best_log_likelihood = float("-inf")

    for s in range(steps + 1):

        lam = s * step
        log_likelihood = 0.0

        for mle in unigram_mles:
            probability = lam * mle + (1 - lam) * uniform
            # Ensure probability is always positive to prevent math domain errors
            probability = max(EPSILON, probability)
            log_likelihood += math.log(probability)

        if log_likelihood > best_log_likelihood:
            best_log_likelihood = log_likelihood
            best_lambda = lam

    if best_lambda is None:
        best_lambda = 0.5

    return best_lambda, best_log_likelihood


def learn_interpolation_weights(dev_sentences, all_counts, all_histories, vocabulary_size):
    """
    Learns, for every order N = 1..MAX_N, the interpolation weights used by
    interpolated_probability(). Returns a dict:
        {1: (lambda,),
         2: (lambda_1, lambda_2),
         3: (lambda_1, lambda_2, lambda_3),
         4: (lambda_1, lambda_2, lambda_3, lambda_4)}
    All tuples are normalized to sum to 1.
    """

    print()
    print("=" * 80)
    print("LEARNING INTERPOLATION WEIGHTS ON THE DEV SET")
    print("=" * 80)

    interpolation_weights = {}

    # N = 1 : unigram formula from the assignment (lambda, 1 - lambda).
    lam, log_likelihood = optimize_unigram_lambda(
        dev_sentences, all_counts, all_histories, vocabulary_size
    )
    interpolation_weights[1] = (lam,)
    print(f"N=1 | lambda = {lam:.3f} "
          f"(1 - lambda = {1 - lam:.3f}) | dev log-likelihood = {log_likelihood:.2f}")

    # N = 2, 3, 4 : full simplex search over (lambda_1, ..., lambda_N).
    for n in range(2, MAX_N + 1):

        component_rows = collect_component_probabilities(
            dev_sentences, n, all_counts, all_histories,
            sample_size=LAMBDA_TUNING_SAMPLE_SIZE
        )

        weights, log_likelihood = optimize_interpolation_weights(component_rows, n)
        interpolation_weights[n] = weights

        weight_text = ", ".join(f"lambda_{k+1}={w:.3f}" for k, w in enumerate(weights))
        print(f"N={n} | {weight_text} | sum={sum(weights):.3f} "
              f"| dev log-likelihood = {log_likelihood:.2f}")

    return interpolation_weights


# INTERPOLATED PROBABILITY
interpolation_cache = {}


def interpolated_probability(ngram, all_counts, all_histories, vocabulary_size, interpolation_weights):

    if ngram in interpolation_cache:
        return interpolation_cache[ngram]

    n = len(ngram)

    # UNIGRAM: P_smooth(w) = lambda * P(w) + (1 - lambda) * (1 / V)
    if n == 1:

        counts = all_counts[1]
        total = all_histories[1][()]
        count = counts.get(ngram, 0)

        mle = count / total if total else 0.0
        uniform = 1.0 / vocabulary_size

        lam = interpolation_weights[1][0]
        probability = lam * mle + (1 - lam) * uniform

        interpolation_cache[ngram] = probability
        return probability

    # N >= 2: standard Jelinek-Mercer linear interpolation across ALL
    # orders 1..n at once, using dev-tuned, normalized weights.
    weights = interpolation_weights[n]

    components = [
        unsmoothed_probability(ngram[-k:], all_counts[k], all_histories[k], k)
        for k in range(1, n + 1)
    ]

    probability = sum(w * c for w, c in zip(weights, components))

    interpolation_cache[ngram] = probability
    return probability


# BUILD KNESER-NEY STATISTICS
def build_kn_statistics(all_counts):

    print("\nBuilding Kneser-Ney statistics...")

    statistics = {}

    for n in range(2, MAX_N + 1):

        print("K-N statistics for N =", n)

        counts = all_counts[n]

        unique_followers = defaultdict(set)
        unique_predecessors = defaultdict(set)

        for ngram in counts:

            history = ngram[:-1]
            word = ngram[-1]

            unique_followers[history].add(word)
            unique_predecessors[word].add(history)

        continuation_counts = {}

        for word in unique_predecessors:
            continuation_counts[word] = len(unique_predecessors[word])

        statistics[n] = {
            "unique_followers": unique_followers,
            "continuation_counts": continuation_counts
        }

        # Free predecessor dictionary.
        del unique_predecessors

        gc.collect()

    return statistics


# KNESER-NEY UNIGRAM PROBABILITIES
def build_kn_unigram_probabilities(kn_statistics):

    continuation_counts = kn_statistics[2]["continuation_counts"]

    total = sum(continuation_counts.values())

    probabilities = {}

    if total == 0:
        return probabilities

    for word, count in continuation_counts.items():
        probabilities[word] = count / total

    return probabilities


# KNESER-NEY PROBABILITY
kn_cache = {}


def kneser_ney_probability(ngram, all_counts, all_histories, kn_statistics, kn_unigram, vocabulary_size):

    if ngram in kn_cache:
        return kn_cache[ngram]

    n = len(ngram)

    # UNIGRAM (continuation probability)
    if n == 1:

        probability = kn_unigram.get(ngram[0], 0.0)

        if probability <= 0.0:
            # A word that never appeared as a continuation of another word in
            # training would otherwise get probability 0, collapsing every
            # higher-order KN estimate that backs off through it and causing
            # infinite perplexity. Fall back to a uniform floor instead.
            probability = 1.0 / vocabulary_size

        kn_cache[ngram] = probability
        return probability

    # CURRENT ORDER
    counts = all_counts[n]
    histories = all_histories[n]
    history = ngram[:-1]

    count = counts.get(ngram, 0)
    history_count = histories.get(history, 0)

    # BACKOFF
    if history_count == 0:
        probability = kneser_ney_probability(
            ngram[1:], all_counts, all_histories, kn_statistics, kn_unigram, vocabulary_size
        )
        kn_cache[ngram] = probability
        return probability

    # FIRST PART
    first_part = max(count - D_VALUE, 0.0) / history_count

    # LAMBDA
    followers = kn_statistics[n]["unique_followers"]
    number_of_followers = len(followers.get(history, ()))
    lambda_history = D_VALUE * number_of_followers / history_count

    # LOWER ORDER
    lower_probability = kneser_ney_probability(
        ngram[1:], all_counts, all_histories, kn_statistics, kn_unigram, vocabulary_size
    )

    probability = first_part + lambda_history * lower_probability
    kn_cache[ngram] = probability

    return probability


# CALCULATE PERPLEXITY WITHOUT STORING N-GRAMS
def calculate_perplexity_from_sentences(sentences, n, probability_function):

    log_sum = 0.0
    total_tokens = 0

    for sentence in sentences:

        padded = prepare_sentence(sentence, n)

        for i in range(len(padded) - n + 1):

            ngram = tuple(padded[i:i + n])
            probability = probability_function(ngram)

            if probability <= 0:
                return float("inf")

            log_sum += math.log(probability)
            total_tokens += 1

    if total_tokens == 0:
        return float("inf")

    return math.exp(-log_sum / total_tokens)


# WRITE N-GRAM CSV


def save_ngram_csv(filename, n, counts, histories):
    print("\nSaving:", filename)

    fieldnames = ["N", "N-gram", "History", "Word", "Count", "History Count"]

    with open(filename, "w", newline="", encoding="utf-8-sig") as file:

        writer = csv.writer(file)
        writer.writerow(fieldnames)

        for ngram, count in counts.items():

            if n == 1:
                history_text = ""
                history_count = histories[()]

            else:
                history_tuple = ngram[:-1]
                history_text = " ".join(history_tuple)
                history_count = histories.get(history_tuple, 0)

            writer.writerow([
                n,
                " ".join(ngram),
                history_text,
                ngram[-1],
                count,
                history_count
            ])

    print("Finished:", filename)


# WRITE SMOOTHED CSV


def save_smoothed_csv(filename, n, counts, histories, vocabulary_size,
                       all_counts, all_histories, kn_statistics, kn_unigram,
                       interpolation_weights):

    print("\nSaving smoothed probabilities:")
    print(filename)

    fieldnames = [
        "N",
        "N-gram",
        "History",
        "Word",
        "Count",
        "History Count",
        "Unsmoothed Probability",
        "Add-One Probability",
        "Add-K Probability",
        "Interpolated Probability",
        "Kneser-Ney Probability"
    ]

    with open(filename, "w", newline="", encoding="utf-8-sig") as file:

        writer = csv.writer(file)
        writer.writerow(fieldnames)

        counter = 0

        for ngram, count in counts.items():

            if n == 1:
                history_text = ""
                history_count = histories[()]

            else:
                history_tuple = ngram[:-1]
                history_text = " ".join(history_tuple)
                history_count = histories.get(history_tuple, 0)

            p_unsmoothed = unsmoothed_probability(ngram, counts, histories, n)
            p_add_one = add_one_probability(ngram, counts, histories, vocabulary_size, n)
            p_add_k = add_k_probability(ngram, counts, histories, vocabulary_size, n)
            p_interpolated = interpolated_probability(
                ngram, all_counts, all_histories, vocabulary_size, interpolation_weights
            )
            p_kn = kneser_ney_probability(
                ngram, all_counts, all_histories, kn_statistics, kn_unigram, vocabulary_size
            )

            writer.writerow([
                n, " ".join(ngram),
                history_text,
                ngram[-1],
                count,
                history_count,
                f"{p_unsmoothed:.12g}",
                f"{p_add_one:.12g}",
                f"{p_add_k:.12g}",
                f"{p_interpolated:.12g}",
                f"{p_kn:.12g}"
            ])

            counter += 1

            if counter % CSV_CHUNK_SIZE == 0:
                print(f"  {counter:,} rows written...")

    print("Finished:", filename)


# MAIN


def main():

    print()
    print("=" * 80)
    print("MEMORY OPTIMIZED N-GRAM LANGUAGE MODEL")
    print("=" * 80)

    print()
    print("Output folder:")
    print(BASE_FOLDER)

    # READ DATA

    print()
    print("Reading training data...")
    train_sentences = read_sentences(TRAIN_FILE)
    print("Training sentences:", len(train_sentences))

    print()
    print("Reading development data...")
    dev_sentences = read_sentences(DEV_FILE)
    print("Development sentences:", len(dev_sentences))

    print()
    print("Reading test data...")
    test_sentences = read_sentences(TEST_FILE)
    print("Test sentences:", len(test_sentences))

    # VOCABULARY

    print()
    print("Building vocabulary...")
    vocabulary = build_vocabulary(train_sentences)
    vocabulary_size = len(vocabulary)
    print("Vocabulary size:", vocabulary_size)

    # UNKNOWN WORDS

    print()
    print("Replacing unknown words...")
    train_sentences = replace_unknown_words(train_sentences, vocabulary)
    dev_sentences = replace_unknown_words(dev_sentences, vocabulary)
    test_sentences = replace_unknown_words(test_sentences, vocabulary)

    del vocabulary
    gc.collect()

    # BUILD MODELS

    # We build all four orders because interpolation and Kneser-Ney need
    # lower-order models.

    all_counts = {}
    all_histories = {}

    for n in range(1, MAX_N + 1):
        counts, histories = build_model(train_sentences, n)
        all_counts[n] = counts
        all_histories[n] = histories

    # K-N STATISTICS

    kn_statistics = build_kn_statistics(all_counts)
    kn_unigram = build_kn_unigram_probabilities(kn_statistics)

    # INTERPOLATION WEIGHTS (learned on dev, normalized to sum to 1)

    interpolation_weights = learn_interpolation_weights(
        dev_sentences, all_counts, all_histories, vocabulary_size
    )

    lambda_rows = []
    for n, weights in interpolation_weights.items():
        for order, w in enumerate(weights, start=1):
            lambda_rows.append({"N": n, "Order": order, "Lambda": w})

    lambda_df = pd.DataFrame(lambda_rows)
    lambda_file = os.path.join(BASE_FOLDER, "interpolation_weights.csv")
    lambda_df.to_csv(lambda_file, index=False, encoding="utf-8-sig")
    print()
    print("Interpolation weights saved:", lambda_file)

    # RESULTS

    results = []

    # PROCESS EACH N

    for n in range(1, MAX_N + 1):

        print()
        print("#" * 80)
        print(f"PROCESSING N = {n}")
        print("#" * 80)

        counts = all_counts[n]
        histories = all_histories[n]

        # SAVE RAW N-GRAM FILE
        ngram_file = os.path.join(BASE_FOLDER, f"N{n}_ngrams.csv")
        save_ngram_csv(ngram_file, n, counts, histories)

        # SAVE SMOOTHED N-GRAM FILE

        interpolation_cache.clear()
        kn_cache.clear()

        smoothed_file = os.path.join(BASE_FOLDER, f"N{n}_smoothed_ngrams.csv")

        save_smoothed_csv(
            smoothed_file,
            n,
            counts,
            histories,
            vocabulary_size,
            all_counts,
            all_histories,
            kn_statistics,
            kn_unigram,
            interpolation_weights
        )

        # PERPLEXITY
        print()
        print("Calculating perplexities...")

        # UNSMOOTHED
        def unsmoothed_function(ngram, counts=counts, histories=histories, n=n):
            return unsmoothed_probability(ngram, counts, histories, n)

        dev_pp = calculate_perplexity_from_sentences(dev_sentences, n, unsmoothed_function)
        test_pp = calculate_perplexity_from_sentences(test_sentences, n, unsmoothed_function)

        print(f"Unsmoothed | Dev = {dev_pp:.4f} | Test = {test_pp:.4f}")

        results.append({"N": n, "Model": "Unsmoothed", "Dev Perplexity": dev_pp, "Test Perplexity": test_pp})

        # ADD ONE
        def add_one_function(ngram, counts=counts, histories=histories, n=n):
            return add_one_probability(ngram, counts, histories, vocabulary_size, n)

        dev_pp = calculate_perplexity_from_sentences(dev_sentences, n, add_one_function)
        test_pp = calculate_perplexity_from_sentences(test_sentences, n, add_one_function)

        print(f"Add-One | Dev = {dev_pp:.4f} | Test = {test_pp:.4f}")

        results.append({"N": n, "Model": "Add-One", "Dev Perplexity": dev_pp, "Test Perplexity": test_pp})

        # ADD K
        def add_k_function(ngram, counts=counts, histories=histories, n=n):
            return add_k_probability(ngram, counts, histories, vocabulary_size, n)

        dev_pp = calculate_perplexity_from_sentences(dev_sentences, n, add_k_function)
        test_pp = calculate_perplexity_from_sentences(test_sentences, n, add_k_function)

        print(f"Add-K | Dev = {dev_pp:.4f} | Test = {test_pp:.4f}")

        results.append({"N": n, "Model": "Add-K", "Dev Perplexity": dev_pp, "Test Perplexity": test_pp})

        # INTERPOLATED (dev-tuned, normalized weights)
        interpolation_cache.clear()

        def interpolated_function(ngram, n=n):
            return interpolated_probability(ngram, all_counts, all_histories, vocabulary_size, interpolation_weights)

        dev_pp = calculate_perplexity_from_sentences(dev_sentences, n, interpolated_function)

        interpolation_cache.clear()

        test_pp = calculate_perplexity_from_sentences(test_sentences, n, interpolated_function)

        print(f"Interpolated | Dev = {dev_pp:.4f} | Test = {test_pp:.4f}")

        results.append({"N": n, "Model": "Interpolated", "Dev Perplexity": dev_pp, "Test Perplexity": test_pp})

        # K-NESER NEY
        kn_cache.clear()

        def kn_function(ngram):
            return kneser_ney_probability(ngram, all_counts, all_histories, kn_statistics, kn_unigram, vocabulary_size)

        dev_pp = calculate_perplexity_from_sentences(dev_sentences, n, kn_function)

        kn_cache.clear()

        test_pp = calculate_perplexity_from_sentences(test_sentences, n, kn_function)

        print(f"Kneser-Ney | Dev = {dev_pp:.4f} | Test = {test_pp:.4f}")

        results.append({"N": n, "Model": "Kneser-Ney", "Dev Perplexity": dev_pp, "Test Perplexity": test_pp})

        # MEMORY CLEANUP

        print()
        print(f"Cleaning temporary memory for N={n}...")

        interpolation_cache.clear()
        kn_cache.clear()

        gc.collect()

        print("Memory cleanup completed.")

    # SAVE PERPLEXITY RESULTS

    print()
    print("=" * 80)
    print("SAVING FINAL RESULTS")
    print("=" * 80)

    results_df = pd.DataFrame(results)

    perplexity_file = os.path.join(BASE_FOLDER, "perplexity_results.csv")
    results_df.to_csv(perplexity_file, index=False, encoding="utf-8-sig")

    print()
    print("Perplexity results saved:")
    print(perplexity_file)

    # PRINT FINAL RESULTS

    print()
    print("=" * 100)
    print("FINAL PERPLEXITY COMPARISON")
    print("=" * 100)
    print(results_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

    # BEST MODEL

    print()
    print("=" * 100)
    print("BEST MODEL FOR EACH N")
    print("=" * 100)

    best_results = []

    for n in range(1, MAX_N + 1):

        subset = results_df[results_df["N"] == n]
        finite = subset[subset["Test Perplexity"].apply(math.isfinite)]

        if finite.empty:
            print(f"N = {n}: No finite result")
            continue

        index = finite["Test Perplexity"].idxmin()
        model = results_df.loc[index, "Model"]
        perplexity = results_df.loc[index, "Test Perplexity"]

        print(f"N = {n} | Best Model = {model} | Test Perplexity = {perplexity:.4f}")

        best_results.append({"N": n, "Best Model": model, "Test Perplexity": perplexity})

    # SAVE BEST MODEL

    best_df = pd.DataFrame(best_results)
    best_file = os.path.join(BASE_FOLDER, "best_models.csv")
    best_df.to_csv(best_file, index=False, encoding="utf-8-sig")

    print()
    print("Best model results saved:")
    print(best_file)

    # CLEAN LARGE TRAINING OBJECTS

    print()
    print("Cleaning large objects from RAM...")

    del train_sentences
    del dev_sentences
    del test_sentences

    del all_counts
    del all_histories

    del kn_statistics
    del kn_unigram

    del results
    del results_df
    del best_results
    del best_df

    interpolation_cache.clear()
    kn_cache.clear()

    gc.collect()

    # CREATE ZIP
    # NOTE: only the generated *.csv result files are archived. The original
    # train/dev/test *.txt corpus files are intentionally excluded - they
    # are the input data, not a result, and including them made the ZIP
    # unnecessarily large.

    print()
    print("=" * 80)
    print("CREATING ZIP FILE")
    print("=" * 80)

    files_to_zip = []

    for filename in os.listdir(BASE_FOLDER):

        full_path = os.path.join(BASE_FOLDER, filename)

        if (os.path.isfile(full_path)
                and filename != os.path.basename(ZIP_FILE)
                and filename.endswith(".csv")):
            files_to_zip.append(full_path)

    with zipfile.ZipFile(ZIP_FILE, "w", compression=zipfile.ZIP_DEFLATED) as zip_file:

        for file_path in files_to_zip:
            zip_file.write(file_path, arcname=os.path.basename(file_path))
            print("Added:", os.path.basename(file_path))

    print()
    print("ZIP created:")
    print(ZIP_FILE)

    # FINAL FILE LIST

    print()
    print("=" * 80)
    print("FILES IN ILLM_LAB_4")
    print("=" * 80)

    for filename in sorted(os.listdir(BASE_FOLDER)):

        full_path = os.path.join(BASE_FOLDER, filename)

        if os.path.isfile(full_path):
            size_mb = os.path.getsize(full_path) / (1024 * 1024)
            print(f"{filename:45s}" f"{size_mb:10.2f} MB")

    # FINAL RAM CLEANUP

    gc.collect()

    # GOOGLE COLAB DOWNLOAD

    try:
        from google.colab import files

        print()
        print("Downloading ZIP file...")
        files.download(ZIP_FILE)

    except Exception:
        print()
        print("Automatic download is not available.")
        print("Your files are saved in:")
        print(BASE_FOLDER)

    # FINISHED

    print()
    print("=" * 80)
    print("PROGRAM FINISHED")
    print("=" * 80)

    print()
    print("All files have been saved to:")
    print(BASE_FOLDER)


# RUN PROGRAM


if __name__ == "__main__":
    main()


MEMORY OPTIMIZED N-GRAM LANGUAGE MODEL

Output folder:
/content/drive/MyDrive/ILLM_LAB_4

Reading training data...
Training sentences: 1000000

Reading development data...
Development sentences: 1000

Reading test data...
Test sentences: 1000

Building vocabulary...
Vocabulary size: 66182

Replacing unknown words...

Building N=1 model...
Unique N=1 grams: 66,181
Histories: 1

Building N=2 model...
Unique N=2 grams: 496,513
Histories: 66,181

Building N=3 model...
Unique N=3 grams: 1,910,766
Histories: 494,140

Building N=4 model...
Unique N=4 grams: 4,957,825
Histories: 1,899,401

Building Kneser-Ney statistics...
K-N statistics for N = 2
K-N statistics for N = 3
K-N statistics for N = 4

LEARNING INTERPOLATION WEIGHTS ON THE DEV SET
N=1 | lambda = 1.000 (1 - lambda = 0.000) | dev log-likelihood = -2517714.18
N=2 | lambda_1=0.500, lambda_2=0.500 | sum=1.000 | dev log-likelihood = -inf
N=3 | lambda_1=0.333, lambda_2=0.333, lambda_3=0.333 | sum=1.000 | dev log-likelihood = -inf
N=4 | l

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


PROGRAM FINISHED

All files have been saved to:
/content/drive/MyDrive/ILLM_LAB_4
